# Week 3 Notebook: Affinity Analysis & Association Rules Mining for Multi-City Booking Data

This notebook adapts the grocery market-basket example to a **multi-city booking** dataset.

## Main idea
In grocery data:
- one basket = one shopping transaction
- one item = one purchased product

In multi-city booking data:
- one basket = one trip (`utrip_id`)
- one item = one visited city (`city_id`)

So we will use association analysis to discover patterns such as:

- trips that include **city A** also often include **city B**
- common combinations of cities across the same trip
- strong rules of the form **A → B**

This version is written in **PySpark / Databricks style** to match the course notebook.


In [1]:
# Verify the Spark version being used in the notebook
print(spark.version)

NameError: name 'spark' is not defined

# 1) Load the booking table

Update `TABLE_NAME` to match the table you created or the table provided to students.

Expected columns include:
- `user_id`
- `checkin`
- `checkout`
- `city_id`
- `device_class`
- `affiliate_id`
- `booker_country`
- `hotel_country`
- `utrip_id`


In [ ]:
from pyspark.sql import functions as F

# Change this table name if your data is stored somewhere else in Databricks
TABLE_NAME = "damo630.default.multicity_booking_data"

# Read the Spark table into a Spark DataFrame
booking_data = spark.table(TABLE_NAME)

# Show a few example rows
display(booking_data.limit(5))

# Print the schema so students can inspect the data types
booking_data.printSchema()

# 2) Understand the dataset

Before building transactions, it is useful to inspect:
- number of rows
- number of trips
- number of users
- number of distinct cities

This helps students understand the scale of the dataset.


In [ ]:
# Basic descriptive counts
n_rows = booking_data.count()
n_trips = booking_data.select("utrip_id").distinct().count()
n_users = booking_data.select("user_id").distinct().count()
n_cities = booking_data.select("city_id").distinct().count()

print("Number of rows:", n_rows)
print("Number of distinct trips (utrip_id):", n_trips)
print("Number of distinct users:", n_users)
print("Number of distinct cities:", n_cities)

# 3) View the most frequent cities

This is similar to looking at the most frequent grocery items.
Here, we count how often each `city_id` appears in the booking records.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

top_cities = (
    booking_data
    .groupBy("city_id")
    .count()
    .orderBy(F.col("count").desc())
    .limit(20)
    .toPandas()
)

plt.figure(figsize=(10, 8))
sns.barplot(data=top_cities, y="city_id", x="count", palette="viridis")
plt.title("Top 20 Most Frequent Cities in Booking Records")
plt.xlabel("Frequency")
plt.ylabel("City ID")
plt.show()

# 4) Data preprocessing: build trip baskets

For market-basket analysis, we need one row per transaction.

Here:
- transaction = one trip (`utrip_id`)
- items = all cities visited during that trip

We use `collect_set(city_id)` so repeated city values inside the same trip are stored once.

We also sort the city list so the basket representation is consistent.


In [ ]:
# Build one basket per trip
transactions = (
    booking_data
    .select(
        F.col("utrip_id"),
        F.col("user_id"),
        F.col("city_id"),
        F.to_date("checkin").alias("checkin"),
        F.to_date("checkout").alias("checkout")
    )
    .filter(F.col("utrip_id").isNotNull())
    .filter(F.col("city_id").isNotNull())
    .groupBy("utrip_id")
    .agg(
        F.first("user_id").alias("user_id"),
        F.min("checkin").alias("trip_start"),
        F.max("checkout").alias("trip_end"),
        F.sort_array(F.collect_set(F.col("city_id").cast("string"))).alias("items")
    )
    .filter(F.size("items") > 0)
)

# Count how many trip baskets were created
n_tx = transactions.count()
print("Number of trip baskets:", n_tx)

# Preview the resulting baskets
display(transactions.limit(10))

# 5) Analyze basket size distribution

Basket size means the number of distinct cities in a trip.

This plot helps answer questions such as:
- Are most trips single-city trips?
- How many trips include multiple cities?
- Is the dataset appropriate for association rule mining?


In [ ]:
basket_pd = (
    transactions
    .withColumn("basket_size", F.size("items"))
    .groupBy("basket_size")
    .count()
    .orderBy("basket_size")
    .toPandas()
)

plt.figure(figsize=(10, 5))
sns.barplot(data=basket_pd, x="basket_size", y="count", color="steelblue")
plt.title("Distribution of Trip Basket Sizes")
plt.xlabel("Number of Distinct Cities in a Trip")
plt.ylabel("Number of Trips")
plt.xticks(basket_pd["basket_size"])
plt.show()

# 6) Average number of cities by trip start month

This optional exploration gives another descriptive view of the data.
You may skip it if the course focuses only on Apriori mechanics.


In [ ]:
monthly_basket_pd = (
    transactions
    .withColumn("basket_size", F.size("items"))
    .withColumn("trip_month", F.month("trip_start"))
    .groupBy("trip_month")
    .agg(F.avg("basket_size").alias("avg_basket_size"))
    .orderBy("trip_month")
    .toPandas()
)

plt.figure(figsize=(10, 4))
sns.barplot(data=monthly_basket_pd, x="trip_month", y="avg_basket_size", palette="Blues")
plt.title("Average Number of Cities per Trip by Start Month")
plt.xlabel("Trip Start Month")
plt.ylabel("Average Basket Size")
plt.show()

# 7) Set Apriori thresholds

## Support
Support measures how frequently an itemset appears in the dataset.

\[
\text{Support}(X)=\frac{\text{Number of transactions containing }X}{\text{Total number of transactions}}
\]

## Confidence
Confidence measures how often a rule \(X \rightarrow Y\) is true.

\[
\text{Confidence}(X \rightarrow Y)=\frac{\text{Support}(X \cup Y)}{\text{Support}(X)}
\]

## Lift
Lift measures whether two items occur together more often than expected by chance.

\[
\text{Lift}(X \rightarrow Y)=\frac{\text{Confidence}(X \rightarrow Y)}{\text{Support}(Y)}
\]


In [ ]:
# These values can be adjusted depending on dataset size and sparsity
minSupport = 0.005
minConfidence = 0.10

print("Minimum support:", minSupport)
print("Minimum confidence:", minConfidence)

# 8) Frequent 1-itemsets

This is the first phase of Apriori.

We explode the basket array so each city appears in its own row, then count city frequency across all trips.


In [ ]:
freq1 = (
    transactions
    .select(F.explode("items").alias("item"))
    .groupBy("item")
    .count()
    .withColumn("support", F.col("count") / F.lit(n_tx))
)

# Keep only cities whose support is above the minimum support threshold
freq1_f = freq1.where(F.col("support") >= minSupport)

print("Number of frequent 1-itemsets:", freq1_f.count())
display(freq1_f.orderBy(F.col("count").desc()).limit(30))

# 9) Frequent 2-itemsets (city pairs)

This is the second phase of Apriori for pairs.

For each trip basket:
- explode the item list twice
- form all pairs `(i, j)`
- keep only `i < j` to avoid duplicates and self-pairs
- count pair frequency across all trips


In [ ]:
pairs = (
    transactions
    .withColumn("i", F.explode("items"))
    .withColumn("j", F.explode("items"))
    .where(F.col("i") < F.col("j"))   # keep unique unordered pairs only
    .select("i", "j")
)

# Preview a few generated city pairs
display(pairs.limit(10))

freq2 = (
    pairs
    .groupBy("i", "j")
    .count()
    .withColumn("support", F.col("count") / F.lit(n_tx))
)

# Keep only frequent pairs
freq2_f = freq2.where(F.col("support") >= minSupport)

print("Number of frequent 2-itemsets:", freq2_f.count())
display(freq2_f.orderBy(F.col("count").desc()).limit(30))

# 10) Prepare frequent 1-itemsets for joins

To compute rule metrics, we need:
- support of the antecedent
- support of the consequent
- support of the pair

So we prepare renamed versions of the 1-itemset table for joining.


In [ ]:
# Statistics for the antecedent X
freq1_stats = freq1.select(
    F.col("item").alias("x"),
    F.col("count").alias("cnt_x"),
    F.col("support").alias("sup_x")
)

# Support for the consequent Y
freq1_support = freq1.select(
    F.col("item").alias("y"),
    F.col("support").alias("sup_y")
)

# 11) Build association rules from frequent pairs

From each frequent pair `(i, j)` we can create two directional rules:
- `i → j`
- `j → i`

For each rule we compute:
- support
- confidence
- lift


In [ ]:
# Rules of the form i -> j
rules_ij = (
    freq2_f
    .join(freq1_stats, freq2_f.i == freq1_stats.x, "left")
    .join(freq1_support, freq2_f.j == freq1_support.y, "left")
    .withColumn("confidence", F.col("count") / F.col("cnt_x"))
    .withColumn("lift", F.col("confidence") / F.col("sup_y"))
    .select(
        F.col("i").alias("antecedent"),
        F.col("j").alias("consequent"),
        "count",
        "support",
        "confidence",
        "lift"
    )
)

# Rules of the form j -> i
rules_ji = (
    freq2_f
    .join(freq1_stats, freq2_f.j == freq1_stats.x, "left")
    .join(freq1_support, freq2_f.i == freq1_support.y, "left")
    .withColumn("confidence", F.col("count") / F.col("cnt_x"))
    .withColumn("lift", F.col("confidence") / F.col("sup_y"))
    .select(
        F.col("j").alias("antecedent"),
        F.col("i").alias("consequent"),
        "count",
        "support",
        "confidence",
        "lift"
    )
)

# 12) Combine and filter the rules

After creating both directions, we combine them and keep only rules above the minimum confidence threshold.


In [ ]:
rules = (
    rules_ij
    .unionByName(rules_ji)
    .where(F.col("confidence") >= minConfidence)
    .orderBy(F.col("confidence").desc(), F.col("lift").desc())
)

print("Number of rules after confidence filtering:", rules.count())
display(rules.limit(50))

# 13) Inspect the strongest rules by lift

### Lift interpretation
- **Lift > 1**: positive association
- **Lift = 1**: independence
- **Lift < 1**: negative association

A high-lift rule suggests that the consequent city appears with the antecedent city more often than expected by chance.


In [ ]:
display(rules.orderBy(F.col("lift").desc()).limit(50))

# 14) Optional: create readable rule text

This is useful for reporting or exporting results to students.


In [ ]:
readable_rules = (
    rules
    .withColumn(
        "rule_text",
        F.concat(
            F.lit("If a trip includes city "),
            F.col("antecedent"),
            F.lit(", it is likely to also include city "),
            F.col("consequent")
        )
    )
)

display(
    readable_rules.select(
        "rule_text", "support", "confidence", "lift"
    ).orderBy(F.col("lift").desc()).limit(20)
)

# 15) Optional export of results

These exports are useful if you want students to save the results as CSV files from Databricks.
You can uncomment and run them if needed.


In [ ]:
# Uncomment these lines if you want to save outputs in Databricks files or tables

# freq1_f.write.mode("overwrite").option("header", True).csv("/FileStore/week3/frequent_1_itemsets")
# freq2_f.write.mode("overwrite").option("header", True).csv("/FileStore/week3/frequent_2_itemsets")
# rules.write.mode("overwrite").option("header", True).csv("/FileStore/week3/association_rules")

# Or save as managed Spark tables:
# freq1_f.write.mode("overwrite").saveAsTable("damo630.default.week3_freq1")
# freq2_f.write.mode("overwrite").saveAsTable("damo630.default.week3_freq2")
# rules.write.mode("overwrite").saveAsTable("damo630.default.week3_rules")

# 16) Questions for students

1. What is the transaction in this dataset?
2. What is the item in this dataset?
3. Why do we use `collect_set()` instead of `collect_list()`?
4. What happens if `minSupport` is set too high?
5. What does it mean if a rule has high confidence but lift close to 1?
6. Which city pairs appear most frequently?
7. Which rules seem most useful for understanding travel patterns?

These questions can be used in class discussion or as a follow-up exercise.
